In [11]:
using Falcons
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")
using Base.Threads
using Statistics
using NPZ

In [12]:
nside_in = 16
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(10.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = 2)
;
cs = ConvolutionSky(lmax = lmax_in, numofsky = 2)
@show fieldnames(typeof(cs))
cb = ConvolutionBeam(lmax = lmax_in, mmax =2, numofbeams = 2)
@show fieldnames(typeof(cb))
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0,)
@show fieldnames(typeof(cc))

fieldnames(typeof(cs)) = (:numofsky, :lmax, :alm)
fieldnames(typeof(cb)) = (:numofbeams, :lmax, :mmax, :blm)
fieldnames(typeof(cc)) = (:nside_output, :resol, :lstart, :lstop, :mmax_calculate, :HWP)


(:nside_output, :resol, :lstart, :lstop, :mmax_calculate, :HWP)

In [15]:
alm_in = npzread("./inputs/alm=CMB=r0=nside16.npy")
alm_in[1,:] .= 0
cs.alm[1,:,:] = alm_in
cs.alm[2,:,:] = alm_in

3×1176 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im          0.0+0.0im  …           0.0+0.0im
 0.0+0.0im  0.0+0.0im   -0.0798721+0.0im        0.0055722+0.00619581im
 0.0+0.0im  0.0+0.0im  -0.00082668+0.0im     -0.000398073+6.20209e-5im

In [16]:
cb.blm[:,:,1] = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)
cb.blm[:,:,2] = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)


141×3 Matrix{ComplexF64}:
 0.282095+0.0im          0.0+0.0im  0.0+0.0im
 0.485926+0.0im          0.0+0.0im  0.0+0.0im
 0.620473+0.0im          0.0+0.0im  0.0+0.0im
 0.722154+0.0im          0.0+0.0im  0.0+0.0im
 0.801049+0.0im          0.0+0.0im  0.0+0.0im
 0.861599+0.0im          0.0+0.0im  0.0+0.0im
 0.906288+0.0im          0.0+0.0im  0.0+0.0im
 0.936785+0.0im          0.0+0.0im  0.0+0.0im
 0.954405+0.0im          0.0+0.0im  0.0+0.0im
 0.960314+0.0im          0.0+0.0im  0.0+0.0im
 0.955628+0.0im          0.0+0.0im  0.0+0.0im
 0.941456+0.0im          0.0+0.0im  0.0+0.0im
 0.918919+0.0im          0.0+0.0im  0.0+0.0im
         ⋮                          
      0.0+0.0im    -0.030715+0.0im  0.0-0.030715im
      0.0+0.0im   -0.0254066+0.0im  0.0-0.0254066im
      0.0+0.0im   -0.0208931+0.0im  0.0-0.0208931im
      0.0+0.0im   -0.0170816+0.0im  0.0-0.0170816im
      0.0+0.0im   -0.0138844+0.0im  0.0-0.0138844im
      0.0+0.0im   -0.0112204+0.0im  0.0-0.0112204im
      0.0+0.0im  -0.00901524

In [17]:
alm_slice = slice_spin_alm_by_l(cs, cc)
blm_slice = slice_spin_blm_by_l(cb, cc);

In [18]:
ss = gen_ScanningStrategy(nside = 32,coord="G")
show_ss(ss)

nside                    : 32 
duration [sec]           : 31536000.0 
sampling rate [Hz]       : 1.0 
alpha [deg]              : 45.0 
beta [deg]               : 50.0 
gamma [deg]              : 0.0 
prec. period [min]       : 192.348
↳ prec. rate [rpm]       : 0.005199
spin period [min]        : 20.000
↳ spin rate [rpm]        : 0.050000
HWP rot. rate[rpm]       : 0.000000 
start angle              : 0.000000 
coordinate system        : G 
FPU                     
↳ Det. 1  name | boresight              : (x,y,z,w) = [0.000, 0.000, 1.000, 0.000] 


In [19]:
nptg = 31536000
pointings_ = zeros(nptg,3)

pointings_[:,1], pointings_[:,2], pointings_[:,3], time_array = get_pointings(ss, 0, nptg)
;

In [20]:
pixels = ang2pixRing.(Ref(cc.resol), pointings_[:,1], pointings_[:,2]);

In [21]:
index_dict = build_index_dict(pixels)


Dict{Int64, Vector{Int64}} with 3072 entries:
  2288 => [9456, 9457, 9458, 9459, 9460, 9461, 9462, 49336, 49337, 49338  …  31…
  1703 => [3934, 3935, 3936, 3937, 3938, 3939, 3940, 3941, 3942, 3943  …  31484…
  1956 => [50736, 50737, 50738, 50739, 50740, 50741, 50742, 50743, 50744, 50745…
  2350 => [38538, 38539, 38540, 38541, 38542, 38543, 38544, 38545, 38546, 38547…
  2841 => [19490, 19491, 19492, 19493, 19494, 19495, 19496, 19497, 19498, 19499…
  2876 => [4580, 4581, 4582, 4583, 4584, 20632, 20633, 20634, 20635, 20636  …  …
  687  => [580, 581, 582, 583, 584, 585, 586, 587, 588, 589  …  31530547, 31530…
  2015 => [13967, 13968, 13969, 13970, 13971, 13972, 13973, 13974, 13975, 13976…
  1090 => [666617, 666618, 666619, 666620, 701396, 701397, 701398, 701399, 7014…
  185  => [9958204, 9958205, 9958206, 9992982, 9992983, 9992984, 9992985, 99929…
  1704 => [46464, 46465, 46466, 46467, 46468, 46469, 46470, 46471, 46472, 46473…
  422  => [11481000, 11515778, 11515779, 11515780, 11538611, 11

In [ ]:
nring = 4*cc.resol.nside - 1
for i in 1:nring
    start, stop = ring_pixel_range(i, cc.resol.nside)
    temp_theta = unique_theta_val(i, cc.resol.nside)
    @time temp_gd = global_wignerd(cc, temp_theta)
    println("Ring $i: pixels $start to $stop")
    for pix in start:stop
        indices = get(index_dict, pix, Int[])
        ang = pix2angRing(cc.resol, pix)
        @time pix_D = global_d2D_conj(cc, temp_gd, ang[2])
        φ = pointings_[indices,2]
        θ = pointings_[indices,1]
        ψ = pointings_[indices,3]
        #@show length(indices)
        @time test_map = compute_pixel_convolution_mapmake(cs, cb, cc, alm_slice, blm_slice, pix, pix_D, φ, θ, ψ; τ=10)
    end
end

  0.209544 seconds (52.06 k allocations: 12.469 MiB, 13.27% compilation time)
Ring 1: pixels 1 to 4
  0.132285 seconds (54.10 k allocations: 12.590 MiB, 22.96% compilation time)
 17.853639 seconds (607.99 k allocations: 43.463 MiB, 0.20% gc time, 1.85% compilation time)
  0.101583 seconds (28 allocations: 8.888 MiB)
 17.320369 seconds (154.84 k allocations: 15.552 MiB)
  0.103041 seconds (28 allocations: 8.888 MiB)
 17.977967 seconds (160.77 k allocations: 15.941 MiB)
  0.101802 seconds (28 allocations: 8.888 MiB)
 17.745373 seconds (159.70 k allocations: 15.871 MiB)
  0.187162 seconds (27 allocations: 8.888 MiB)
Ring 2: pixels 5 to 12
  0.101444 seconds (28 allocations: 8.888 MiB)
 16.994066 seconds (154.75 k allocations: 15.546 MiB)
  0.100437 seconds (28 allocations: 8.888 MiB)
 17.518642 seconds (155.60 k allocations: 15.601 MiB)
  0.103017 seconds (28 allocations: 8.888 MiB)
 16.995682 seconds (153.93 k allocations: 15.491 MiB)
  0.103015 seconds (28 allocations: 8.888 MiB)
 17.45